In [2]:
# Feature engineering
#Setup
from pathlib import Path

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA = ROOT / "data"

# print(ROOT)

In [3]:
# Import cleaned CSV files
import pandas as pd

# print(pd.read_csv(DATA / "cleaned" / "price_cleaned.csv", nrows=5))
# print(pd.read_csv(DATA / "cleaned" / "price_cleaned.csv", nrows=0).columns.tolist())


price = pd.read_csv(
    DATA / "cleaned" / "price_cleaned.csv",
    parse_dates=["date"],
    index_col="date"
)

onchain = pd.read_csv(
    DATA / "cleaned" / "onchain_cleaned.csv",
    parse_dates=["date"],
    index_col="date"
)

In [4]:
# 1, 7, and 30 day returns
### R_t=(P_t-P_(t-1))/P_(t-1) 
###     R_t = return on current day
###     P_t= Closing price on current day
###     P_(t-1)= Closing price on previous day
### Provides daily percentage change

price["return_1d"] = price["close"].pct_change(fill_method=None)

price["return_7d"] = price["close"].pct_change(periods=7, fill_method=None)

price["return_30d"] = price["close"].pct_change(periods=30, fill_method=None)


In [5]:
# Running averages
# 7 & 30 day moving averages

price["ma_7d"] = price["close"].rolling(7).mean()
price["ma_30d"] = price["close"].rolling(30).mean()


In [6]:
# Rolling volatility -- measures magnitude of recent plrice fluctuation using std. dev.
price["volatility_7d"] = (price["return_1d"].rolling(7).std())

price["volatility_30d"] = (price["return_1d"].rolling(30).std())


In [7]:
# Previous days returns -- lagged returns
price["return_lag_1"] = price["return_1d"].shift(1)
price["return_lag_2"] = price["return_1d"].shift(2)

price["price_range"] = ((price["high"] - price["low"]) / price["open"])

In [8]:
# Onchain feature engineering
import numpy as np

metrics = [
    "active_addresses",
    "transaction_count",
    "hash_rate",
    "transaction_fees"
]

for metric in metrics:
    onchain[f"{metric}_change_7d"] = (
        onchain[metric].pct_change(
            periods=7,
            fill_method=None
        )
    )

# Replace infinite values w/missing values
onchain = onchain.replace(
    [np.inf, -np.inf],
    np.nan
)

In [9]:
# Account for on-chain reportign delays to reduce look-ahead bias. 
# On-chain metrics available on day t are used to predict close
# price direction on day t + 1
# On-chain features are shifted

onchain = onchain.shift(1)

In [10]:
# Create target variable (binary classification)
next_close = price["close"].shift(-1)

price["target"] = (
    next_close > price["close"]
).astype("Int64")

# Final observation excluded if following day's closing price is unknown
price.loc[
    next_close.isna() | price["close"].isna(),
    "target"
] = pd.NA

# Retain next-day return for trading strategy evaluation
price["next_day_return"] = (next_close / price["close"] - 1)



In [11]:
# Merge datasets

from math import comb


combined = price.join(
    onchain,
    how="left",
    validate="one_to_one"
)

# Inspect resulting datasets
print(combined.head())
print(combined.shape)
print(combined.isna().sum())

# Restrict dataset to common modeling period
combined = combined.loc[
    "2018-01-01":"2026-05-23"
].copy()

# Define feature columns
price_features = [
    col for col in price.columns
    if col not in ["target", "next_day_return"]
]

onchain_features = [
    col for col in onchain.columns
    if col not in ["target", "next_day_return"]
]

# Enhanced dataset: price and on-chain features
enhanced = combined[
    price_features + onchain_features + ["target"]
].dropna().copy()

# Baseline dataset: price features only
baseline = combined[
    price_features + ["target"]
].loc[enhanced.index].copy()

# Verify datasets
print("Baseline shape: ", baseline.shape)
print("Enhanced shape: ", enhanced.shape)

print("\nBaseline features:")
print(baseline.columns.tolist())

print("\nEnhanced features:")
print(enhanced.columns.tolist())

print(baseline.isna().sum())
print(enhanced.isna().sum())

print(baseline.index.equals(enhanced.index))

print("Baseline:", baseline.shape)
print("Enhanced:", enhanced.shape)

# print(combined.isna().sum())

                              open     high      low    close  return_1d  \
date                                                                       
2017-11-13 00:00:00+00:00  5839.94  6697.47  5699.99  6465.99        NaN   
2017-11-14 00:00:00+00:00  6465.99  6684.98  6311.07  6574.99   0.016857   
2017-11-15 00:00:00+00:00  6575.99  7298.00  6575.99  7240.06   0.101151   
2017-11-16 00:00:00+00:00  7240.14  7940.00  7076.00  7864.50   0.086248   
2017-11-17 00:00:00+00:00  7876.98  7989.00  7451.00  7699.19  -0.021020   

                           return_7d  return_30d  ma_7d  ma_30d  \
date                                                              
2017-11-13 00:00:00+00:00        NaN         NaN    NaN     NaN   
2017-11-14 00:00:00+00:00        NaN         NaN    NaN     NaN   
2017-11-15 00:00:00+00:00        NaN         NaN    NaN     NaN   
2017-11-16 00:00:00+00:00        NaN         NaN    NaN     NaN   
2017-11-17 00:00:00+00:00        NaN         NaN    NaN     NaN  

In [ ]:
# Verify
# print(combined.head())
# print(combined.tail())
# print(combined.shape)
# print(combined.index.min())
# print(combined.index.max())
# print(combined.isna().sum())
# print("Start:", baseline.index.min())
# print("End:", baseline.index.max())

In [ ]:
# Split dataset chronologically
# Training 2018 - 2024
# Validation 2025
# Testing 2026 year to date

# Since target uses following day's closing price, exclude 12-31 from training & validation, preventing labels from extending into next sets

# Baseline splits
baseline_train = baseline.loc["2018-01-01":"2024-12-30"].copy()
baseline_validation = baseline.loc["2025-01-01":"2025-12-30"].copy()
baseline_test = baseline.loc["2026-01-01":].copy()

# Enhanced splits
enhanced_train = enhanced.loc["2018-01-01":"2024-12-30"].copy()
enhanced_validation = enhanced.loc["2025-01-01":"2025-12-30"].copy()
enhanced_test = enhanced.loc["2026-01-01":].copy()

# combined = combined.loc["2018-01-01":].dropna()
# train = combined.loc["2018-01-01":"2024-12-30"].copy()
# validation = combined.loc["2025-01-01":"2025-12-30"].copy()
# test = combined.loc["2026-01-01":].copy()


In [ ]:
# Save processed data sets to csv

processed = DATA / "processed"

baseline_dir = processed / "baseline"
enhanced_dir = processed / "enhanced"

baseline_dir.mkdir(parents=True, exist_ok=True)
enhanced_dir.mkdir(parents=True, exist_ok=True)

# Baseline
baseline_train.to_csv(baseline_dir / "train.csv")
baseline_validation.to_csv(baseline_dir / "validation.csv")
baseline_test.to_csv(baseline_dir / "test.csv")

# Enhanced
enhanced_train.to_csv(enhanced_dir / "train.csv")
enhanced_validation.to_csv(enhanced_dir / "validation.csv")
enhanced_test.to_csv(enhanced_dir / "test.csv")

In [ ]:
# Verification of sets

print("                 Baseline    Enhanced")
print("Training:      ", baseline_train.shape, enhanced_train.shape)
print("Validation:    ", baseline_validation.shape, enhanced_validation.shape)
print("Testing:       ", baseline_test.shape, enhanced_test.shape)

                 Baseline    Enhanced
Training:       (2556, 15) (2556, 24)
Validation:     (364, 15) (364, 24)
Testing:        (143, 15) (143, 24)


In [ ]:
print("On-chain start: ", onchain.index.min())
print("On-chain end: ", onchain.index.max())

print("Baseline start: ", baseline.index.min())
print("Baseline end: ", baseline.index.max())


On-chain start:  2017-12-01 00:00:00+00:00
On-chain end:  2026-05-23 00:00:00+00:00
Baseline start:  2018-01-01 00:00:00+00:00
Baseline end:  2026-05-23 00:00:00+00:00
